In [1]:
import numpy as np
import pennylane as qml
import torch
import torch.nn as nn
from pathlib import Path
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving figures
import matplotlib.pyplot as plt

# Ensure output directory exists
output_dir = Path("results/figures")
output_dir.mkdir(parents=True, exist_ok=True)

In [7]:
dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def circuit_identity():
    return qml.state()

@qml.qnode(dev)
def circuit_flip():
    qml.PauliX(wires=0)
    return qml.state()

@qml.qnode(dev)
def circuit_hadamard():
    qml.Hadamard(wires=0)
    return qml.state()

circuit_identity()
circuit_flip()
circuit_hadamard()

array([0.70710678+0.j, 0.70710678+0.j])

In [22]:
@qml.qnode(dev)
def circuit_ry(theta):
    qml.RY(theta, wires=0)
    return qml.expval(qml.PauliZ(0))

thetas = np.linspace(0, 2*np.pi, 50)
expectations = [circuit_ry(theta) for theta in thetas]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thetas, expectations, 'b-', linewidth=2)
ax.set_xlabel('θ (radians)', fontsize=12)
ax.set_ylabel('⟨Z⟩ (expectation value)', fontsize=12)
ax.set_title('RY(θ) gate: How rotation angle affects measurement', fontsize=14)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=np.pi, color='red', linestyle='--', alpha=0.5, label='θ=π (|1⟩)')
ax.axvline(x=np.pi/2, color='green', linestyle='--', alpha=0.5, label='θ=π/2 (|+⟩)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(output_dir / "ry_sweep.png", dpi=150)
print(f"\n  Plot saved to: {output_dir / 'ry_sweep.png'}")

print(f"\n  Key observations:")
print(f"    θ = 0:   ⟨Z⟩ = {circuit_ry(0.0):.3f}  (qubit in |0⟩)")
print(f"    θ = π/2: ⟨Z⟩ = {circuit_ry(np.pi/2):.3f}  (qubit in |+⟩)")
print(f"    θ = π:   ⟨Z⟩ = {circuit_ry(np.pi):.3f}  (qubit in |1⟩)")


  Plot saved to: results/figures/ry_sweep.png

  Key observations:
    θ = 0:   ⟨Z⟩ = 1.000  (qubit in |0⟩)
    θ = π/2: ⟨Z⟩ = 0.000  (qubit in |+⟩)
    θ = π:   ⟨Z⟩ = -1.000  (qubit in |1⟩)


In [27]:
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def angle_encoding_circuit(inputs):
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

test_vectors = [
    np.array([0.0, 0.0, 0.0, 0.0]),           # All zeros
    np.array([np.pi, np.pi, np.pi, np.pi]),     # All π
    np.array([0.5, 1.0, 1.5, 2.0]),            # Increasing
    np.array([np.pi/4, np.pi/2, 3*np.pi/4, np.pi]),  # Quarter turns
]

print("\n  Input Vector              →  Measurement Output")
print("  " + "-" * 56)
for v in test_vectors:
    output = angle_encoding_circuit(v)
    v_str = "[" + ", ".join(f"{x:.2f}" for x in v) + "]"
    o_str = "[" + ", ".join(f"{x:.3f}" for x in output) + "]"
    print(f"  {v_str:28s} →  {o_str}")

print("""
  Observation:
    - Input [0,0,0,0] → Output [1,1,1,1]  (all qubits in |0⟩)
    - Input [π,π,π,π] → Output [-1,-1,-1,-1]  (all qubits in |1⟩)
    - The output is cos(input) for each qubit!
    - This is because ⟨Z⟩ after RY(θ) = cos(θ)
""")


  Input Vector              →  Measurement Output
  --------------------------------------------------------
  [0.00, 0.00, 0.00, 0.00]     →  [1.000, 1.000, 1.000, 1.000]
  [3.14, 3.14, 3.14, 3.14]     →  [-1.000, -1.000, -1.000, -1.000]
  [0.50, 1.00, 1.50, 2.00]     →  [0.878, 0.540, 0.071, -0.416]
  [0.79, 1.57, 2.36, 3.14]     →  [0.707, 0.000, -0.707, -1.000]

  Observation:
    - Input [0,0,0,0] → Output [1,1,1,1]  (all qubits in |0⟩)
    - Input [π,π,π,π] → Output [-1,-1,-1,-1]  (all qubits in |1⟩)
    - The output is cos(input) for each qubit!
    - This is because ⟨Z⟩ after RY(θ) = cos(θ)



In [28]:
n_qubits = 4
n_layers = 2
dev4 = qml.device("default.qubit", wires=n_qubits)

torch.manual_seed(42)
weights = torch.randn(n_layers, n_qubits, requires_grad=True)
sample_input = torch.tensor([0.5, 1.0, 1.5, 2.0])

In [29]:
weights

tensor([[ 0.3367,  0.1288,  0.2345,  0.2303],
        [-1.1229, -0.1863,  2.2082, -0.6380]], requires_grad=True)

In [30]:
sample_input

tensor([0.5000, 1.0000, 1.5000, 2.0000])